# Solving problems using Python

[Colab notebook](https://colab.research.google.com/drive/1M1UZ2X9bZK6DV_puEH5WVtisVoYt1iMQ?usp=sharing)


We will solve LP problems using Python with scipy.optimize and numpy instead of Gurobi.

In this course we will use scipy.optimize to solve optimization problems. Scipy is an open-source library that provides efficient numerical routines including optimization.
 
For more information, see:
* https://docs.scipy.org/doc/scipy/reference/optimize.html
* https://numpy.org/doc/stable/


In [ ]:
# Install required packages
!pip install scipy numpy matplotlib

In [ ]:
# Import libraries
import numpy as np
from scipy.optimize import linprog, minimize
import warnings
warnings.filterwarnings('ignore')

## Example 1: Simple Factory Problem

Maximize: $40x_1 + 50x_2$

Subject to:
- $x_1 + 2x_2 \leq 40$ (wood)
- $4x_1 + 3x_2 \leq 120$ (labor)
- $x_1, x_2 \geq 0$

In [ ]:
# Define the problem using scipy.optimize.linprog
# Note: linprog minimizes by default, so we negate the objective for maximization
c = [-40, -50]  # Objective coefficients (negated)

# Inequality constraints: A_ub @ x <= b_ub
A_ub = [[1, 2],   # wood
        [4, 3]]   # labor
b_ub = [40, 120]

# Variable bounds
bounds = [(0, None), (0, None)]

# Solve
result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')

print(f"chairs: {result.x[0]:.1f}")
print(f"tables: {result.x[1]:.1f}")
print(f"optimal total revenue: {-result.fun:.1f}")

Given the constraints, the maximum revenue is $1360 structured such that we produce 24 chairs and 8 tables.

## Scalable Formulation

When LP problems have many variables and constraints, we should use data structures and numpy arrays for efficient computation.

### Using dictionaries and lists

In [ ]:
#List comprehension examples
sqrd = [i*i for i in range(10)]
print(sqrd) 

bigsqrd = [i*i for i in range(10) if i*i >= 5]
print(bigsqrd) 

prod = [i+j for i in range(3) for j in range(3)]
print(prod) 

sum_sq = sum(i for i in range(11))
print(sum_sq)

In [ ]:
# Resource data
resources = ['wood', 'labor']
capacity = {'wood': 40, 'labor': 120}
print(resources, capacity)

In [ ]:
# Products data
products = ['chair', 'table']
price = {'chair': 40, 'table': 50}
print(products, price)

In [ ]:
# Bill of materials
bom = {
    ('wood', 'chair'): 1,
    ('wood', 'table'): 2,
    ('labor', 'chair'): 4,
    ('labor', 'table'): 3
}
print(bom)

In [ ]:
# Build constraint matrix using data structures
c = [-price[p] for p in products]  # Objective (negated for max)

# Build A matrix
A_ub = []
b_ub = []
for r in resources:
    row = [bom[(r, p)] for p in products]
    A_ub.append(row)
    b_ub.append(capacity[r])

bounds = [(0, None) for _ in products]

# Solve
result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')

# Display results
for i, p in enumerate(products):
    print(f"{p}: {result.x[i]:.1f}")
print(f"optimal total revenue: {-result.fun:.1f}")

# Sensitivity analysis of LP problems

Solving LP problems provides more information than only the values of the decision variables and the value of the objective function. However, scipy.optimize.linprog provides limited sensitivity analysis compared to commercial solvers. For detailed sensitivity analysis, consider using scipy.optimize.linprog with the `revised simplex` method or examining the dual values.

In [ ]:
# Solve with revised simplex to get more information
result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')

print("Decision variables:")
for i, p in enumerate(products):
    print(f"  {p} = {result.x[i]:.1f}")
print(f"\nObjective value: {-result.fun:.1f}")

# Note: scipy doesn't provide sensitivity ranges like commercial solvers
# For production use, consider using CVXPY or PuLP for more detailed sensitivity info

### Adding new variable

In [ ]:
# Add bench as a new product
products = ['chair', 'table', 'bench']
price = {'chair': 40, 'table': 50, 'bench': 30}

bom = {
    ('wood', 'chair'): 1,
    ('wood', 'table'): 2,
    ('wood', 'bench'): 1.2,
    ('labor', 'chair'): 4,
    ('labor', 'table'): 3,
    ('labor', 'bench'): 2,
}

# Rebuild and solve
c = [-price[p] for p in products]
A_ub = [[bom[(r, p)] for p in products] for r in resources]
b_ub = [capacity[r] for r in resources]
bounds = [(0, None) for _ in products]

result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')

for i, p in enumerate(products):
    print(f"{p}: {result.x[i]:.1f}")
print(f"optimal total revenue: {-result.fun:.1f}")

### Adding new constraint

In [ ]:
# Add packaging constraint
products = ['chair', 'table']
price = {'chair': 40, 'table': 50}

resources = ['wood', 'labor', 'packaging']
capacity = {'wood': 40, 'labor': 120, 'packaging': 5}

bom = {
    ('wood', 'chair'): 1,
    ('wood', 'table'): 2,
    ('labor', 'chair'): 4,
    ('labor', 'table'): 3,
    ('packaging', 'chair'): 0.2,
    ('packaging', 'table'): 0.1,
}

c = [-price[p] for p in products]
A_ub = [[bom[(r, p)] for p in products] for r in resources]
b_ub = [capacity[r] for r in resources]
bounds = [(0, None) for _ in products]

result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')

for i, p in enumerate(products):
    print(f"{p}: {result.x[i]:.1f}")
print(f"optimal total revenue: {-result.fun:.1f}")

## Problem example: C2Q1
Solve the following LP problem:

$$
\begin{align}
&\text{max}\\
&\qquad z=10x_1+6x_2\\
&\text{s.t.}\\
&\qquad 3x_1+8x_2\le 20\\
&\qquad 45x_1+30x_2\le 180\\
&\qquad x_1, x_2\ge 0\\
\end{align} 
$$

In [ ]:
c = [-10, -6]  # Negated for maximization
A = [[3, 8],
     [45, 30]]
b = [20, 180]

result = linprog(c, A_ub=A, b_ub=b, bounds=[(0, None), (0, None)], method='highs')

print(f"x0 = {result.x[0]:.1f}")
print(f"x1 = {result.x[1]:.1f}")
print(f"objective value = {-result.fun:.1f}")

## Problem example: C2Q2
Solve the following LP problem:

$$
\begin{align}
&\text{min}\\
&\qquad z=0.5x_1+0.03x_2\\
&\text{s.t.}\\
&\qquad 8x_1+6x_2\ge 48\\
&\qquad x_1+2x_2\ge 12\\
&\qquad x_1, x_2\ge 0\\
\end{align} 
$$

In [ ]:
c = [0.5, 0.03]  # Minimization (no negation needed)

# For >= constraints, we use A_ub with negated coefficients
# Or use -A_ub @ x <= -b_ub which is equivalent to A_ub @ x >= b_ub
A_ub = [[-8, -6],   # Negate for >= constraint
        [-1, -2]]
b_ub = [-48, -12]   # Negate for >= constraint

result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=[(0, None), (0, None)], method='highs')

print(f"x0 = {result.x[0]:.1f}")
print(f"x1 = {result.x[1]:.1f}")
print(f"objective value = {result.fun:.2f}")

## Problem example: C4Q8
Solve the following LP problem:

$$
\begin{align}
&\text{min}\\
&\qquad z=4x_1+3x_2+2x_3\\
&\text{s.t.}\\
&\qquad 2x_1+4x_2+x_3\ge 16\\
&\qquad 3x_1+2x_2+x_3\ge 12\\
&\qquad x_1, x_2, x_3\ge 0\\
\end{align} 
$$

In [ ]:
c = [4, 3, 2]
A_ub = [[-2, -4, -1],
        [-3, -2, -1]]
b_ub = [-16, -12]

result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=[(0, None)] * 3, method='highs')

print(f"x0 = {result.x[0]:.1f}")
print(f"x1 = {result.x[1]:.1f}")
print(f"x2 = {result.x[2]:.1f}")
print(f"objective value = {result.fun:.1f}")

## Problem example: C4Q32
Solve the following LP problem (Assignment Problem):

$$
\begin{align}
&\text{min}\\
&\qquad z=22x_1+18x_2+35x_3+41x_4+30x_5+28x_6+25x_7+36x_8+18x_9\\
&\text{s.t.}\\
&\qquad x_1+x_2+x_3=1\\
&\qquad x_4+x_5+x_6=1\\
&\qquad x_7+x_8+x_9=1\\
&\qquad x_1+x_4+x_7=1\\
&\qquad x_2+x_5+x_8=1\\
&\qquad x_3+x_6+x_9=1\\
&\qquad x_1,x_2,x_3,x_4,x_5,x_6,x_7,x_8,x_9\ge 0\\
\end{align} 
$$

In [ ]:
c = [22, 18, 35, 41, 30, 28, 25, 36, 18]

# Equality constraints: A_eq @ x = b_eq
A_eq = [[1, 1, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 1, 1, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 1, 1, 1],
        [1, 0, 0, 1, 0, 0, 1, 0, 0],
        [0, 1, 0, 0, 1, 0, 0, 1, 0],
        [0, 0, 1, 0, 0, 1, 0, 0, 1]]
b_eq = [1, 1, 1, 1, 1, 1]

result = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=[(0, None)] * 9, method='highs')

for i in range(9):
    print(f"x{i} = {result.x[i]:.1f}")
print(f"objective value = {result.fun:.1f}")

## Problem example: C4Q33 (Transportation Problem)

Solve the following LP problem:

$$
\begin{align}
&\text{min}\\
&\qquad z=40x_1 + 65x_2 + 70x_3 + 30x_4\\
&\text{s.t.}\\
&\qquad x_1+x_2=250\\
&\qquad x_3+x_4=400\\
&\qquad x_1+x_3=300\\
&\qquad x_2+x_4=350\\
&\qquad x_1,x_2,x_3,x_4\ge 0\\
\end{align} 
$$

In [ ]:
c = [40, 65, 70, 30]
A_eq = [[1, 1, 0, 0],
        [0, 0, 1, 1],
        [1, 0, 1, 0],
        [0, 1, 0, 1]]
b_eq = [250, 400, 300, 350]

result = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=[(0, None)] * 4, method='highs')

for i in range(4):
    print(f"x{i} = {result.x[i]:.1f}")
print(f"objective value = {result.fun:.1f}")